# Aula 03 — Pydantic e Validacao em Runtime

Neste notebook, exploramos como proteger **fronteiras de sistema**
(APIs, workers, pipelines) validando payloads antes de processar inferencia.

## Conceitos-chave

| Conceito | Descricao |
|----------|----------|
| **Schema Validation** | Verificacao estrutural de tipos e campos obrigatorios |
| **Business Rules** | Regras de negocio que vao alem do schema |
| **Strategy Pattern** | Trocar o backend de validacao sem mudar o contrato |
| **Chain of Responsibility** | Compor validadores pequenos e independentes |

## Diferenca entre validacao de DataFrame e validacao de payload

- **Pandera** valida **batches** (DataFrames inteiros)
- **Pydantic** valida **registros individuais** (payloads de API)
- Ambos sao complementares em uma pipeline de ML

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

## 1. Strategy Pattern — Schema Validation

O Strategy Pattern permite trocar o backend de validacao sem mudar
o codigo que consome o resultado. Quando Pydantic esta disponivel,
usamos; senao, fallback local.

In [ ]:
from pydantic_validation import (
    select_schema_strategy,
    LocalSchemaStrategy,
    PydanticSchemaStrategy,
)

strategy = select_schema_strategy()
print(f"Backend selecionado: {strategy.name}")
print(f"Tipo: {type(strategy).__name__}")

In [ ]:
# Testar validacao com payload valido
valid_payload = {
    "customer_id": "C-100",
    "age": 41,
    "monthly_spend": 245.8,
    "prediction_horizon_days": 30,
    "segment": "plus",
}

errors = strategy.validate(valid_payload)
print(f"Erros no payload valido: {errors}")
print(f"Aceito: {not errors}")

In [ ]:
# Testar com payload invalido
invalid_payload = {
    "customer_id": "",
    "age": "quarenta",  # tipo errado
    "monthly_spend": -10,
    # prediction_horizon_days ausente
}

errors = strategy.validate(invalid_payload)
print(f"Erros no payload invalido: {errors}")

## 2. Chain of Responsibility — Business Rules

Alem do schema, precisamos validar **regras de negocio**:
- Faixas operacionais do modelo (idade entre 18 e 100)
- Segmentos permitidos
- Consistencia entre campos

A cadeia permite adicionar novos checks sem modificar os existentes.

In [ ]:
from pydantic_validation import build_validation_chain

chain = build_validation_chain()

# Payload com violacao de faixa
range_payload = {
    "customer_id": "C-101",
    "age": 15,           # abaixo do minimo operacional (18)
    "monthly_spend": 120.0,
    "prediction_horizon_days": 120,  # acima do maximo (90)
    "segment": "basic",
}

business_errors = chain.handle(range_payload)
print(f"Erros de negocio: {business_errors}")

In [ ]:
# Payload com segmento invalido
segment_payload = {
    "customer_id": "C-102",
    "age": 35,
    "monthly_spend": 200.0,
    "prediction_horizon_days": 30,
    "segment": "enterprise",  # nao existe
}

errors = chain.handle(segment_payload)
print(f"Erros: {errors}")

## 3. Pipeline completa: Schema + Business Rules

A funcao `validate_payload` combina schema validation e business
rules em uma unica chamada.

In [ ]:
from pydantic_validation import validate_payload, build_sample_payloads

payloads = build_sample_payloads()

for name, payload in payloads.items():
    result = validate_payload(name, payload)
    status = "PASS" if result.accepted else "FAIL"
    print(f"[{status}] {result.payload_name}")
    print(f"        backend={result.schema_backend}")
    if result.errors:
        print(f"        erros={list(result.errors)}")
    print()

## 4. Visualizando a cadeia de validadores

A Chain of Responsibility forma uma sequencia onde cada handler
executa seu check e passa para o proximo.

In [ ]:
from pydantic_validation import (
    RequiredBusinessFieldsHandler,
    NumericRangeHandler,
    AllowedSegmentHandler,
    ConsistencyHandler,
)

# Mostrar a cadeia
handlers = [
    ("1. RequiredBusinessFields", RequiredBusinessFieldsHandler),
    ("2. NumericRange", NumericRangeHandler),
    ("3. AllowedSegment", AllowedSegmentHandler),
    ("4. Consistency", ConsistencyHandler),
]

print("Cadeia de validadores:")
for label, handler_class in handlers:
    print(f"  {label} -> {handler_class.__doc__.strip().split(chr(10))[0]}")

## Resumo

- **Strategy Pattern** permite trocar backend de validacao (Pydantic vs local)
- **Chain of Responsibility** compoe validadores sem acoplar logica
- Schema validation cuida de **tipos e campos**
- Business rules cuidam de **restricoes operacionais**
- O mesmo padrao serve para APIs, jobs de scoring e workers